<a id="top"></a>
# House Sales: preparazione dati e primo modello (esercitazione)

Nel modulo 1 hai usato il dataset House Sales per inquadrare il problema di business: stimare il prezzo di vendita delle case in un’area specifica.  
Nel modulo 2 sul marketing bancario hai invece lavorato sulla preparazione del dataset per un modello di classificazione, curando tipi, encoding, imputazione e normalizzazione.

Qui riutilizzi il caso House Sales come laboratorio tecnico: l’obiettivo non è cambiare il problema di business, ma allenarti a rifare lo stesso flusso di preparazione dati, questa volta su un dominio diverso, e collegarlo a un modello lineare con feature normalizzate.

<a id="indice"></a>
## Indice

- [Obiettivo dell'esercitazione](#obiettivo-esercitazione)
- [Caricamento del dataset House Sales](#caricamento-dataset)
- [Definizione del target binario e scelta delle feature](#target-feature)
- [Encoding delle variabili categoriali](#encoding-categoriche)
- [Imputazione e normalizzazione](#imputazione-scaling)
- [Costruzione del dataset finale per il modello](#dataset-finale)
- [Primo modello lineare: LogisticRegression](#modello-logistic)
- [Spunti di verifica](#spunti-verifica)


<a id="obiettivo-esercitazione"></a>
## Obiettivo dell'esercitazione

Nel caso House Sales il problema naturale resta un task di **regressione**: prevedere un prezzo continuo per ogni casa.  
Per esercitarti con la pipeline vista sul marketing bancario, qui costruirai invece un problema di **classificazione binaria** derivato dal prezzo: distinguere le case di fascia alta da tutte le altre.

L’obiettivo tecnico è chiaro:

- partire dal DataFrame grezzo `house` scaricato da OpenML;
- definire un target binario collegato al prezzo;
- preparare una matrice di feature numeriche priva di valori mancanti;
- applicare una normalizzazione robusta alle feature non binarie;
- addestrare un primo modello lineare `LogisticRegression` sulle feature scalate.

<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Nota importante**

Il modello di classificazione "casa di fascia alta sì/no" è una semplificazione deliberata.  
Serve a lavorare con una `LogisticRegression` sensibile alla scala delle variabili, sfruttando lo stesso schema **imputazione → scaling robusto → modello lineare** già visto nel modulo 2 sul marketing.  
Per il business reale tornerai a ragionare sul problema di regressione del prezzo.

</div>


<a id="caricamento-dataset"></a>
## Caricamento del dataset House Sales

Userai lo stesso dataset del modulo 1: le vendite di case nella contea di King (Seattle e dintorni).  
Il dataset è disponibile su OpenML e si può caricare con `fetch_openml` di scikit-learn.

- `fetch_openml` scarica un dataset da OpenML e restituisce un oggetto con i dati e la descrizione.  
  Documentazione: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_openml.html

Nel modulo 1 hai già visto come costruire il DataFrame `house`. Qui ripeti rapidamente il passaggio per avere un punto di partenza chiaro.


In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml

# Scarica il dataset House Sales da OpenML
raw = fetch_openml(data_id=42092, as_frame=True)  # House sales in King County
house = raw.frame.copy()

print("Dimensioni del dataset (righe, colonne):", house.shape)
house.head()


<a id="target-feature"></a>
## Definizione del target binario e scelta delle feature

La colonna del prezzo (`price`) resta il cuore del problema di business.  
Per usare una `LogisticRegression` hai però bisogno di un target binario. Qui definisci quindi:

- una soglia di prezzo basata sulla **mediana** dei prezzi;
- un target `is_high_price` uguale a 1 se il prezzo è maggiore o uguale alla soglia, 0 altrimenti.

Userai:

- `Series.median()` per calcolare la mediana di una colonna numerica.  
  Documentazione: https://pandas.pydata.org/docs/reference/api/pandas.Series.median.html
- `Series.value_counts()` per verificare la distribuzione del nuovo target.  
  Documentazione: https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html

<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Esercizio**

Nel blocco di codice seguente:

1. completa il nome della colonna del prezzo dove vedi `___`;  
2. usa il metodo `median()` per calcolare la soglia di prezzo;  
3. crea il target binario `is_high_price` a partire dalla soglia.

Poi osserva la distribuzione di `is_high_price`: le classi sono sbilanciate? quanto?

</div>


In [ ]:
house_prep = house.copy()

# Colonna del prezzo
price_col = "___"  # TODO: inserisci il nome corretto della colonna del prezzo (es. "price")

# Soglia di prezzo per definire le case di fascia alta
price_threshold = house_prep[price_col].___()  # TODO: usa il metodo median()

print("Soglia di prezzo (mediana):", price_threshold)

# Target binario: 1 = casa con prezzo >= soglia, 0 altrimenti
house_prep["is_high_price"] = (house_prep[price_col] >= price_threshold).astype("int8")

# Controllo rapido della distribuzione del target
house_prep["is_high_price"].value_counts(normalize=True)


<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Attenzione**

Il target `is_high_price` deriva direttamente da `price`.  
Quando costruisci il modello, **price non può restare tra le feature**: sarebbe una fuga di informazione banale.  
Nel passaggio successivo togli dalle feature sia `price` sia le colonne puramente identificative, come `id` o `date` se presenti.

</div>


In [ ]:
target_col = "is_high_price"

# Colonne da escludere dalle feature (se presenti)
cols_to_drop = [price_col, "id", "date"]
cols_to_drop = [c for c in cols_to_drop if c in house_prep.columns]

feature_cols = [c for c in house_prep.columns if c not in cols_to_drop + [target_col]]

X = house_prep[feature_cols].copy()
y = house_prep[target_col].copy()

print("Numero di feature candidate:", len(feature_cols))
X.head()


<a id="encoding-categoriche"></a>
## Encoding delle variabili categoriali

Nel dataset House Sales quasi tutte le colonne sono numeriche.  
L’unica variabile chiaramente **nominale** è `zipcode`, che rappresenta la zona della casa e non ha un ordine naturale.

Per renderla utilizzabile da un modello lineare:

- identifichi `zipcode` come colonna nominale;
- applichi un **one-hot encoding** con `pandas.get_dummies`, rimuovendo una categoria di riferimento.

Richiamo:

- `pandas.get_dummies` espande le colonne categoriali in colonne 0/1.  
  Documentazione: https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html

<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Esercizio**

Nel codice seguente:

1. verifica che `zipcode` sia effettivamente presente tra le colonne;  
2. controlla la forma del DataFrame prima e dopo l’encoding e ragiona su quante nuove feature vengono create.

</div>


In [ ]:
X_enc = X.copy()

# Colonne nominali da espandere con one-hot encoding (se presenti)
nominal_cols = [col for col in ["zipcode"] if col in X_enc.columns]

print("Colonne nominali da espandere:", nominal_cols)

X_enc = pd.get_dummies(
    X_enc,
    columns=nominal_cols,
    drop_first=True  # rimuove una categoria di riferimento per ogni variabile
)

print("Forma delle feature dopo l'encoding:", X_enc.shape)
X_enc.head()


In [ ]:
# Controllo dei valori mancanti dopo l'encoding
X_enc.isna().mean().sort_values(ascending=False).head(10)


<a id="imputazione-scaling"></a>
## Imputazione e normalizzazione

Per collegare questo caso al flusso visto sul marketing bancario ripeti gli stessi due passaggi chiave:

1. **imputazione** dei valori mancanti sulle feature numeriche;
2. **normalizzazione robusta** delle feature non binarie con `RobustScaler`.

Strumenti già visti nel modulo 2 Bank Marketing:

- `IterativeImputer` (`sklearn.impute.IterativeImputer`) per stimare i valori mancanti usando modelli sulle altre colonne.  
  Documentazione: https://scikit-learn.org/stable/modules/generated/sklearn.impute.IterativeImputer.html
- `RobustScaler` (`sklearn.preprocessing.RobustScaler`) per scalare le feature usando mediana e IQR, più stabile in presenza di outlier.  
  Documentazione: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html

<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Esercizio**

Leggi con attenzione il codice di imputazione e normalizzazione qui sotto e collega ogni blocco al concetto visto nel modulo 2:  
quale parte corrisponde al `fit`, quale al `transform`, dove vengono ricostruiti i DataFrame con i nomi di colonna.

</div>


In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer

# Imputazione dei valori mancanti
imputer = IterativeImputer(random_state=42)

X_imputed_array = imputer.fit_transform(X_enc)

# Ricostruiamo un DataFrame per mantenere nomi di colonne e indice
X_imputed = pd.DataFrame(
    X_imputed_array,
    columns=X_enc.columns,
    index=X_enc.index
)

# Controllo rapido: nessun valore mancante
X_imputed.isna().sum().sum()


In [ ]:
from sklearn.preprocessing import RobustScaler
import numpy as np

# Identificazione delle colonne binarie (solo valori 0/1)
binary_cols = []
for col in X_imputed.columns:
    unique_vals = pd.unique(X_imputed[col])
    if set(unique_vals).issubset({0, 1, 0.0, 1.0}):
        binary_cols.append(col)

numeric_cols = [col for col in X_imputed.columns if col not in binary_cols]

print("Numero di feature binarie lasciate inalterate:", len(binary_cols))
print("Numero di feature scalate:", len(numeric_cols))

scaler = RobustScaler()

X_scaled = X_imputed.copy()
X_scaled[numeric_cols] = scaler.fit_transform(X_imputed[numeric_cols])

X_scaled.head()


<a id="dataset-finale"></a>
## Costruzione del dataset finale per il modello

A questo punto hai:

- `X_imputed`: feature numeriche senza valori mancanti, non scalate;
- `X_scaled`: le stesse feature, con scaling robusto sulle colonne non binarie;
- `y`: target binario `is_high_price`.

Come nel caso marketing costruisci due DataFrame modellabili:

- `df_model` con le feature imputate non scalate;
- `df_model_scaled` con le feature scalate robustamente.

Nel resto dell’esercitazione userai `df_model_scaled` per addestrare la `LogisticRegression`.


In [ ]:
df_model = X_imputed.copy()
df_model[target_col] = y

df_model_scaled = X_scaled.copy()
df_model_scaled[target_col] = y

print("Forma df_model (non scalato):", df_model.shape)
print("Forma df_model_scaled (scalato):", df_model_scaled.shape)
df_model.head()


<a id="modello-logistic"></a>
## Primo modello lineare: LogisticRegression

Per chiudere l’esercitazione costruisci un primo modello di classificazione binaria su `is_high_price` usando una `LogisticRegression`:

- usi le feature **scalate** in `df_model_scaled` (i coefficienti e l’ottimizzazione dipendono dalla scala);
- separi il dataset in train e test con `train_test_split`;
- addestri il modello e valuti le prestazioni con `classification_report`.

Riferimenti:

- `sklearn.linear_model.LogisticRegression`  
  Documentazione: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
- `sklearn.model_selection.train_test_split`  
  Documentazione: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
- `sklearn.metrics.classification_report`  
  Documentazione: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html

<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Esercizio**

Nel blocco di codice seguente:

1. completa il nome della colonna target da rimuovere da `X`;  
2. imposta `test_size=0.2` per usare il 20% dei dati come test set;  
3. imposta `max_iter=1000` nella `LogisticRegression` per dare al solver tempo sufficiente a convergere.

Dopo l’addestramento:
- leggi il `classification_report` e confronta precision e recall per le due classi;
- ragiona su che tipo di errori sono più frequenti: case costose scambiate per non costose o viceversa?

</div>


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import numpy as np

# Matrice delle feature e target (dataset scalato)
X_final = df_model_scaled.drop(columns=["___"])  # TODO: inserisci il nome della colonna target (es. "is_high_price")
y_final = df_model_scaled["___"]                # TODO: stesso nome della colonna target

# Suddivisione train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y_final,
    test_size=___,    # TODO: usa 0.2 per il 20% di test
    random_state=42
)

# Modello lineare con scaling robusto a monte
log_clf = LogisticRegression(
    max_iter=___,     # TODO: usa 1000
    n_jobs=-1
)

log_clf.fit(X_train, y_train)

y_pred = log_clf.predict(X_test)

print("Report di classificazione sul test set:\n")
print(classification_report(y_test, y_pred))

# Previsioni di esempio con probabilità di "casa di fascia alta"
y_proba = log_clf.predict_proba(X_test)[:, 1]

preview = pd.DataFrame({
    "y_true": y_test.iloc[:10].values,
    "y_pred": y_pred[:10],
    "p_high_price": np.round(y_proba[:10], 3)
})

preview


<a id="spunti-verifica"></a>
## Spunti di verifica

Alcune domande utili per consolidare il lavoro svolto:

- In che cosa questo flusso "preparazione dati → modello" assomiglia a quello del marketing bancario?  
- In che cosa è diverso, dal punto di vista del problema (regressione sul prezzo vs classificazione casa di fascia alta)?  
- Guardando il `classification_report`, come si comporta il modello sulle case di fascia alta rispetto alle altre?  
- Se volessi tornare a un problema di regressione sul prezzo, quali parti della preparazione potresti riutilizzare così come sono?

Queste domande ti preparano ai moduli successivi, dove la scelta del modello e delle metriche verrà collegata in modo più esplicito alle priorità del business.
